# Teste de Extração: PAM (Produção Agrícola Municipal)
Extrai dados de lavouras temporárias (1612) e permanentes (1613) via API SIDRA.

In [ ]:
import os
import logging
import pandas as pd
import sidrapy
import uuid
import time
from pathlib import Path
from tqdm import tqdm
from abc import ABC, abstractmethod
from typing import List, Generator
from concurrent.futures import ThreadPoolExecutor, as_completed

# Configuração de Caminhos
DATA_DIR = Path("data")
BRONZE_DIR = DATA_DIR / "bronze"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

class EscritorParquet:
    def __init__(self, diretorio_base: Path):
        self.diretorio_base = diretorio_base

    def escrever_chunk(self, df: pd.DataFrame, nome_dataset: str, coluna_particao: str = None):
        if df is None or df.empty: return
        diretorio_destino = self.diretorio_base / nome_dataset
        if coluna_particao and coluna_particao in df.columns:
            valor_particao = df[coluna_particao].iloc[0]
            diretorio_destino = diretorio_destino / f"{coluna_particao}={valor_particao}"
            df = df.drop(columns=[coluna_particao])
        diretorio_destino.mkdir(parents=True, exist_ok=True)
        df.to_parquet(diretorio_destino / f"chunk_{uuid.uuid4().hex[:8]}.parquet", index=False)

class ExtratorPAM:
    def __init__(self, anos: List[int]):
        self.anos = anos

    def _baixar_ano(self, ano: int) -> pd.DataFrame:
        dfs = []
        for tab, tipo in [("1612", "temporaria"), ("1613", "permanente")]:
            try:
                d = sidrapy.get_table(table_code=tab, territorial_level="6", ibge_territorial_code="all", period=str(ano))
                if d is not None and len(d) > 1:
                    d = d.iloc[1:].copy()
                    d["tipo_lavoura"] = tipo
                    dfs.append(d)
                time.sleep(0.5)
            except Exception as e: print(f"Erro {tab} {ano}: {e}")
        return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    def extrair(self):
        with ThreadPoolExecutor(max_workers=3) as ex:
            futures = {ex.submit(self._baixar_ano, a): a for a in self.anos}
            for f in tqdm(as_completed(futures), total=len(self.anos), desc="Baixando PAM"): yield f.result()

anos = list(range(2020, 2024))
extrator = ExtratorPAM(anos)
escritor = EscritorParquet(BRONZE_DIR)

for chunk in extrator.extrair():
    if not chunk.empty:
        # Simplificação para teste: usando D1C como ano se presente
        part = "D1C" if "D1C" in chunk.columns else None
        escritor.escrever_chunk(chunk, "pam", coluna_particao=part)
print("Concluído!")